In [8]:

# ─── CORRECT imports ──────────────────────────────────────────────────────────
# langchain_core.stores.InMemoryStore  →  document/KV store (different API)
# langgraph.store.memory.InMemoryStore →  LangGraph persistent store (what we need)
# LangGraph injection only triggers on langgraph.store.base.BaseStore type hint
from langgraph.store.memory import InMemoryStore   # FIXED: was langchain_core.stores
from langgraph.store.base import BaseStore         # FIXED: was langchain_core.stores
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from typing import TypedDict, List, Annotated
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages

load_dotenv()
ssl_cert_file = "C:\\Users\\Ujjwal_Manna\\Downloads\\ca-certs.crt"
client = httpx.Client(verify=ssl.create_default_context(cafile=ssl_cert_file))

client_id = os.environ.get('LLM_CLIENT_ID')
client_secret = os.environ.get('LLM_CLIENT_SECRET')
auth_header = f"Basic {base64.b64encode(f'{client_id}:{client_secret}'.encode()).decode()}"

llm_openai = ChatOpenAI(
    model="gpt-oss-120b",
    base_url=os.environ.get("LLM_URL"),
    http_client=client,
    api_key="dummy value",
    default_headers={
        'accept': '*/*',
        'Content-Type': 'application/json',
        'Authorization': auth_header
    },
    temperature=0.0,
    streaming=True
)

# ─── Pydantic schema for structured fact extraction ───────────────────────────
class UserProfile(BaseModel):
    name: str = Field(default="", description="User's name if mentioned, else empty string")
    company: str = Field(default="", description="User's company if mentioned, else empty string")
    products: List[str] = Field(default=[], description="ABC products user mentioned using")
    found_any: bool = Field(default=False, description="True only if at least one field was extracted")

# ─── State ────────────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    # add_messages reducer: APPENDS new messages, never overwrites
    messages: Annotated[List, add_messages]
    user_id: str  # used as the key in the store — same user_id = same profile across all threads

# ─── Node 1: Extract facts from latest user message → save to store ───────────
# NOTE: LangGraph injects 'store' automatically because the type hint is
#       langgraph.store.base.BaseStore (NOT langchain_core.stores.BaseStore)
#       This injection only works when graph.compile(store=store) is called.
def save_facts_node(state: AgentState, store: BaseStore) -> dict:
    """
    Q1 Answer: Why can't the checkpointer alone do cross-thread memory?
    Checkpointer is SCOPED to thread_id — new thread = empty slate.
    Store is GLOBAL — not bound to any thread. One user_id → one profile forever.
    """
    user_id = state.get("user_id", "")
    if not user_id:
        return {}

    # Only inspect the latest human message
    human_messages = [m for m in state["messages"] if isinstance(m, HumanMessage)]
    if not human_messages:
        return {}

    latest = human_messages[-1].content

    # Use structured output to extract facts from natural language
    extractor = llm_openai.with_structured_output(UserProfile)
    extracted: UserProfile = extractor.invoke([
        SystemMessage(content=(
            "Extract user profile facts from the message. "
            "Only extract what is explicitly stated. "
            "Set found_any=True only if you found at least one field."
        )),
        HumanMessage(content=latest)
    ])

    if not extracted.found_any:
        return {}

    """
    Q2 Answer: Why is namespace a tuple, not a string?
    Tuples allow HIERARCHICAL namespacing like a directory tree:
      ("users",)            → all users (can search/list all)
      ("users", user_id)    → all data for one specific user
    store.search(("users",)) returns ALL items under "users" regardless of key.
    A flat string can't express hierarchy or enable prefix-based searches.
    """
    # Load existing profile to MERGE — don't overwrite with empty strings
    existing_item = store.get(("users",), user_id)
    existing: dict = existing_item.value if existing_item else {}

    merged = {
        "name":     extracted.name    or existing.get("name", ""),
        "company":  extracted.company or existing.get("company", ""),
        "products": list(set(existing.get("products", []) + extracted.products))
    }

    store.put(("users",), user_id, merged)
    print(f"[store] Saved profile for '{user_id}': {merged}")

    return {}  # No state change — profile lives in store, not in messages

# ─── Node 2: Load profile from store → inject as context → chat ───────────────
def chat_node(state: AgentState, store: BaseStore) -> dict:
    user_id = state.get("user_id", "")

    # Load profile from store — works across ALL threads for this user_id
    profile_item = store.get(("users",), user_id) if user_id else None
    profile: dict = profile_item.value if profile_item else {}

    # Build enriched system message from profile
    base_system = (
        "You are a helpful ABC customer support agent named Aria. "
        "Remember everything the user tells you within this conversation."
    )
    if profile and any(profile.values()):
        profile_context = "\n\nUser profile on file:"
        if profile.get("name"):
            profile_context += f"\n- Name: {profile['name']}"
        if profile.get("company"):
            profile_context += f"\n- Company: {profile['company']}"
        if profile.get("products"):
            profile_context += f"\n- Products used: {', '.join(profile['products'])}"
        system_msg = SystemMessage(content=base_system + profile_context)
    else:
        system_msg = SystemMessage(content=base_system)

    response: AIMessage = llm_openai.invoke([system_msg] + state["messages"])
    return {"messages": [response]}

# ─── Graph: START → save_facts → chat → END ───────────────────────────────────
graph = StateGraph(AgentState)
graph.add_node("save_facts", save_facts_node)
graph.add_node("chat", chat_node)
graph.add_edge(START, "save_facts")
graph.add_edge("save_facts", "chat")
graph.add_edge("chat", END)

store = InMemoryStore()      # global persistent store (cross-thread)
checkpoint = MemorySaver()   # per-thread short-term memory

# CRITICAL: pass store= so LangGraph can inject it into node signatures
app = graph.compile(checkpointer=checkpoint, store=store)

print(app.get_graph().draw_mermaid())


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	save_facts(save_facts)
	chat(chat)
	__end__([<p>__end__</p>]):::last
	__start__ --> save_facts;
	save_facts --> chat;
	chat --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pyppeteer\us_keyboard_layout.py:73: SyntaxWarning: invalid escape sequence '\('
  'Digit9': {'keyCode': 57, 'code': 'Digit9', 'shiftKey': '\(', 'key': '9'},
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pyppeteer\us_keyboard_layout.py:143: SyntaxWarning: invalid escape sequence '\<'
  'Comma': {'keyCode': 188, 'code': 'Comma', 'shiftKey': '\<', 'key': ','},
C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pyppeteer\us_keyboard_layout.py:247: SyntaxWarning: invalid escape sequence '\<'
  '<': {'keyCode': 188, 'key': '\<', 'code': 'Comma'},


In [9]:

# ─── Cross-thread memory test ─────────────────────────────────────────────────
def chat_with_user(thread_id: str, user_id: str, user_input: str):
    result = app.invoke(
        {"messages": [HumanMessage(content=user_input)], "user_id": user_id},
        config={"configurable": {"thread_id": thread_id}}
    )
    print(f"  AI: {result['messages'][-1].content}\n")

print("=== SESSION 1 — thread: session_001, user: ujjwal ===")
chat_with_user("session_001", "ujjwal", "Hi, my name is Ujjwal. I work at ABC and use a Latitude laptop.")

print("=== SESSION 2 — NEW thread: session_002, same user: ujjwal ===")
print("    (MemorySaver has ZERO memory of session_001 — but store does)")
chat_with_user("session_002", "ujjwal", "What laptop do I use?")

print("=== SESSION 3 — different user: alice — must NOT know about Ujjwal ===")
chat_with_user("session_003", "alice", "What is my name and what company do I work at?")

print("=== STORE INSPECTION ===")
all_users = store.search(("users",))
for item in all_users:
    print(f"  user_id={item.key}  profile={item.value}")


=== SESSION 1 — thread: session_001, user: ujjwal ===


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=UserProfile(name='Ujjwal'...aptop'], found_any=True), input_type=UserProfile])
  return self.__pydantic_serializer__.to_python(


[store] Saved profile for 'ujjwal': {'name': 'Ujjwal', 'company': 'Dell', 'products': ['Latitude laptop']}
  AI: Hello Ujjwal! 👋  

Welcome back to Dell Support. I’m Aria, here to help you with your Latitude laptop. How can I assist you today?

=== SESSION 2 — NEW thread: session_002, same user: ujjwal ===
    (MemorySaver has ZERO memory of session_001 — but store does)


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=UserProfile(name='', comp...cts=[], found_any=False), input_type=UserProfile])
  return self.__pydantic_serializer__.to_python(


  AI: Hi Ujjwal! Based on the information we have on file, you’re using a **Dell Latitude laptop**. If you need details about the specific model or have any other questions, just let me know!

=== SESSION 3 — different user: alice — must NOT know about Ujjwal ===


C:\projects\personal\langchn_practise\.venv\Lib\site-packages\pydantic\main.py:475: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=UserProfile(name='', comp...cts=[], found_any=False), input_type=UserProfile])
  return self.__pydantic_serializer__.to_python(


  AI: I’m sorry—I don’t have that information on hand. Could you let me know your name and the company you’re with? That way I can make sure I’m addressing you correctly and help you with any Dell‑related questions you have.

=== STORE INSPECTION ===
  user_id=ujjwal  profile={'name': 'Ujjwal', 'company': 'Dell', 'products': ['Latitude laptop']}
